# Marketing Mix Modeling: from-scratch model vs Google Meridian

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzacharrad29-png/mmm-budget-optimizer/blob/main/mmm_colab.ipynb)

This notebook clones the project from GitHub, runs both models, compares them and downloads a zip with all results.

**Before you start**
1. **Runtime > Change runtime type > T4 GPU > Save.** Meridian needs the GPU.
2. **Runtime > Run all.** Total time: about 20-30 minutes (Meridian is the slow part).
3. At the end, your browser downloads `mmm-results.zip` (the `outputs/` and `reports/` folders).

If Colab shows a **Restart session** button after the install cell, click it, then **Run all** again.

## 1. Get the code

In [ ]:
import os
if not os.path.exists("/content/mmm-budget-optimizer"):
    !git clone -q https://github.com/hamzacharrad29-png/mmm-budget-optimizer.git /content/mmm-budget-optimizer
%cd /content/mmm-budget-optimizer

## 2. Install the dependencies and check the GPU

In [ ]:
!pip install -q -r requirements.txt "google-meridian[colab,schema]"
import jax
print("JAX devices:", jax.devices())
if jax.default_backend() != "gpu":
    print("WARNING: no GPU found. Go to Runtime > Change runtime type > T4 GPU, then Run all again.")

## 3. From-scratch model (about 2 minutes)
Geometric adstock + Hill saturation + bounded regression, fitted on national totals and on the 40 regions.

In [ ]:
%run train.py

## 4. Google Meridian (about 15-25 minutes on a T4 GPU)
Bayesian model on regional data. Saves ROI with 90% credible intervals, the budget recommendation and HTML reports.

In [ ]:
%run run_meridian.py

## 5. Compare the two models

In [ ]:
%run compare.py

## 6. Meridian reports
The full reports are in the `reports/` folder of the downloaded zip (open them in a browser). Preview of the summary:

In [ ]:
from pathlib import Path
from IPython.display import HTML
report = Path("reports/summary_output.html")
HTML(report.read_text()) if report.exists() else print("Report not found: run section 4 first.")

## 7. Download the results

In [ ]:
!rm -f /content/mmm-results.zip && zip -rq /content/mmm-results.zip outputs reports
from google.colab import files
files.download("/content/mmm-results.zip")

## Run the Streamlit app

The interactive budget simulator (`app.py`) runs on your own computer, not inside Colab. It only needs the `model_params.json` files in `outputs/regional/` and `outputs/national/`, which are already in the repository, so you do not need a GPU or Meridian for it.

**1. Clone the repository and open the folder**
```bash
git clone https://github.com/hamzacharrad29-png/mmm-budget-optimizer.git
cd mmm-budget-optimizer
```

**2. Create a virtual environment and install the dependencies** (Python 3.10 or newer)
```bash
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

**3. (Optional) Re-train the from-scratch model.** This regenerates `outputs/national/` and `outputs/regional/` in about 2 minutes:
```bash
python train.py
```

**4. Start the app**
```bash
streamlit run app.py
```
Your browser opens at http://localhost:8501. Stop the app with `Ctrl+C` in the terminal.

**Using the app**
- **Model:** the regional model (default) or the national model.
- **Total weekly budget (% of current):** scale the total budget from 60% to 140% of today's spend.
- **Max change per channel (+/- %):** how far each channel may move from its current spend.
- The metrics compare media-driven revenue with the current mix versus the optimized mix for the same budget.
- The table shows the recommended spend, ROI and marginal ROI per channel; the response-curve chart shows where the current and optimized spend sit on each channel's saturation curve.

**Updating the repository with new results.** After running this notebook, unzip `mmm-results.zip` into your local copy of the repository (replacing `outputs/` and `reports/`), then commit and push.